In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("BusDisruptionProject")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

print("PySpark version:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/11 23:29:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


PySpark version: 3.5.1


## libraries

In [2]:
from pathlib import Path
from datetime import datetime, timezone
from getpass import getpass
import xml.etree.ElementTree as ET

import requests

print("Libraries imported successfully.")

Libraries imported successfully.


## project folder

In [3]:
current_folder = Path.cwd()
if current_folder.name == "notebooks":
    project_root = current_folder.parent
else:
    project_root = current_folder
print("Current folder:", current_folder)
print("Project root:", project_root)

Current folder: /Users/babitaadhikari/Desktop/bus-disruption-platform/notebooks
Project root: /Users/babitaadhikari/Desktop/bus-disruption-platform


## vehicle-location folder

In [4]:
location_folder = (
    project_root
    / "data"
    / "raw"
    / "vehicle_location"
)
location_folder.mkdir(
    parents=True,
    exist_ok=True
)
print("Vehicle-location folder:")
print(location_folder)

Vehicle-location folder:
/Users/babitaadhikari/Desktop/bus-disruption-platform/data/raw/vehicle_location


## the feed URL

In [5]:
feed_url = getpass(
    "https://data.bus-data.dft.gov.uk/api/v1/datafeed/10609/?api_key=e57c5f929b40320c65e3bb87a9c3500161663a56"
)

if not feed_url.strip():
    raise ValueError("The feed URL cannot be empty.")

print("Feed URL received securely.")

https://data.bus-data.dft.gov.uk/api/v1/datafeed/10609/?api_key=e57c5f929b40320c65e3bb87a9c3500161663a56 ········


Feed URL received securely.


## live snapshot

In [6]:
try:
    response = requests.get(
        feed_url.strip(),
        timeout=120
    )
    print("Status code:", response.status_code)
    print(
        "Content type:",
        response.headers.get(
            "Content-Type",
            "Unknown"
        )
    )
    print("Downloaded bytes:", len(response.content))
    response.raise_for_status()
except requests.RequestException as error:
    raise RuntimeError(
        f"Download failed: {error}"
    )

Status code: 200
Content type: text/xml
Downloaded bytes: 1282777


In [7]:
content_start = response.content.lstrip()[:100]

if not content_start.startswith(b"<"):
    print(response.text[:500])
    raise ValueError(
        "The downloaded response is not XML."
    )

print("The downloaded response appears to be XML.")

The downloaded response appears to be XML.


## save the XML file

In [8]:
timestamp = datetime.now(
    timezone.utc
).strftime("%Y%m%dT%H%M%SZ")

output_file = location_folder / (
    f"nx_west_midlands_{timestamp}.xml"
)

output_file.write_bytes(
    response.content
)

print("Vehicle-location data downloaded successfully.")
print("Saved file:", output_file)
print(
    "File size:",
    round(
        output_file.stat().st_size / 1_000_000,
        2
    ),
    "MB"
)

Vehicle-location data downloaded successfully.
Saved file: /Users/babitaadhikari/Desktop/bus-disruption-platform/data/raw/vehicle_location/nx_west_midlands_20260711T174953Z.xml
File size: 1.28 MB


In [9]:
try:
    tree = ET.parse(output_file)
    root = tree.getroot()

    print("XML parsed successfully.")
    print("Root element:", root.tag)

except ET.ParseError as error:
    raise ValueError(
        f"Invalid XML file: {error}"
    )

XML parsed successfully.
Root element: {http://www.siri.org.uk/siri}Siri


In [10]:
def remove_namespace(tag):
    return tag.split("}")[-1]


vehicle_records = [
    element
    for element in root.iter()
    if remove_namespace(element.tag)
    == "VehicleActivity"
]

print(
    "VehicleActivity records:",
    len(vehicle_records)
)

VehicleActivity records: 1146
